# 09 - Distributed Training on Snowflake

This notebook demonstrates Snowflake's distributed training capabilities for scaling ML workloads across multiple nodes and GPUs.

**Three approaches**:
1. **Snowflake ML Estimators** (XGBClassifier, LightGBMClassifier) -- distributed on warehouse
2. **PyTorchDistributor** -- multi-node GPU training on compute pools
3. **ML Jobs** -- remote execution on dedicated compute pools

**Also covered**: DataConnector API for efficient data loading.

**Prerequisites**: Run `00_data_setup.ipynb` first. GPU compute pools needed for PyTorch examples.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print(f"Connected: {session.get_current_user()}")

## DataConnector API

The DataConnector provides optimized, distributed data ingestion from Snowflake tables directly into ML framework datasets. Significantly faster than `.to_pandas()` for large datasets.

Supports conversion to:
- pandas DataFrames
- PyTorch Datasets
- TensorFlow Datasets

In [ ]:
from snowflake.ml.data import DataConnector

# Efficient data loading from Snowflake tables
connector = DataConnector.from_dataframe(
    session.table("MLOPS_DEMO_DB.RAW.CUSTOMERS")
)

# Convert to pandas (optimized path)
pdf = connector.to_pandas()
print(f"Loaded {len(pdf)} rows via DataConnector")
print(f"Columns: {list(pdf.columns)}")

In [ ]:
# Convert to PyTorch Dataset (for deep learning pipelines)
# Requires torch to be installed
try:
    torch_ds = connector.to_torch_dataset(
        batch_size=64,
        shuffle=True
    )
    batch = next(iter(torch_ds))
    print(f"PyTorch batch shape: {batch}")
except Exception as e:
    print(f"PyTorch Dataset: {e}")
    print("Install torch in the notebook environment for PyTorch integration.")

## Distributed XGBoost with Snowflake ML Estimators

Snowflake ML provides drop-in replacements for common ML estimators that automatically distribute computation across warehouse nodes. Same API as scikit-learn, but parallelized.

In [ ]:
from snowflake.ml.modeling.xgboost import XGBClassifier as SnowXGBClassifier

# Load data as Snowpark DataFrame (stays in Snowflake)
train_df = session.table("MLOPS_DEMO_DB.RAW.CUSTOMERS")

feature_cols = [
    "CREDIT_SCORE", "AGE", "TENURE", "BALANCE",
    "NUM_OF_PRODUCTS", "HAS_CR_CARD", "IS_ACTIVE_MEMBER", "ESTIMATED_SALARY"
]

# Distributed XGBoost -- automatically parallelized
xgb_model = SnowXGBClassifier(
    input_cols=feature_cols,
    label_cols=["EXITED"],
    output_cols=["PREDICTED_CHURN"],
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1
)

print("Training distributed XGBoost...")
xgb_model.fit(train_df)
print("Training complete.")

In [ ]:
# Run predictions (also distributed)
predictions = xgb_model.predict(train_df)
print("Distributed predictions:")
predictions.select("CUSTOMER_ID", "PREDICTED_CHURN").show(10)

In [ ]:
# LightGBM is also available as a distributed estimator
from snowflake.ml.modeling.lightgbm import LGBMClassifier as SnowLGBMClassifier

lgbm_model = SnowLGBMClassifier(
    input_cols=feature_cols,
    label_cols=["EXITED"],
    output_cols=["PREDICTED_CHURN_LGBM"],
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1
)

print("Training distributed LightGBM...")
lgbm_model.fit(train_df)
lgbm_preds = lgbm_model.predict(train_df)
print("LightGBM predictions:")
lgbm_preds.select("CUSTOMER_ID", "PREDICTED_CHURN_LGBM").show(5)

## PyTorchDistributor for Deep Learning

For deep learning workloads, PyTorchDistributor enables multi-node, multi-GPU training on compute pools. The framework handles:
- Automatic resource discovery
- DistributedDataParallel (DDP) setup
- Ray-backed cluster management

**Note**: Requires an active GPU compute pool (ML_GPU_POOL).

In [ ]:
# PyTorchDistributor example (conceptual -- requires GPU compute pool)
#
# from snowflake.ml.modeling.distributors import PyTorchDistributor
#
# def train_fn():
#     import torch
#     import torch.nn as nn
#     from torch.utils.data import DataLoader
#
#     # Define model
#     model = nn.Sequential(
#         nn.Linear(8, 64),
#         nn.ReLU(),
#         nn.Dropout(0.3),
#         nn.Linear(64, 32),
#         nn.ReLU(),
#         nn.Dropout(0.3),
#         nn.Linear(32, 1),
#         nn.Sigmoid()
#     )
#
#     # Training loop with DDP
#     optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
#     criterion = nn.BCELoss()
#
#     for epoch in range(10):
#         # ... training loop ...
#         pass
#
#     return model
#
# distributor = PyTorchDistributor(
#     num_processes=4,
#     compute_pool="ML_GPU_POOL"
# )
# trained_model = distributor.run(train_fn)

print("PyTorchDistributor enables multi-node GPU training.")
print("Supported GPU types: GPU_NV_S, GPU_NV_M, GPU_NV_L")
print("Frameworks: PyTorch, with automatic DDP setup.")

## ML Jobs - Remote Execution on Compute Pools

ML Jobs allow running Python ML scripts on dedicated compute pools, separate from the notebook's compute. This is useful for:
- Long-running training jobs
- GPU-intensive workloads
- Scheduled execution via Snowflake Tasks

In [ ]:
# ML Jobs example (conceptual -- requires active compute pool)
#
# from snowflake.ml.jobs import remote
#
# @remote(
#     compute_pool="ML_GPU_POOL",
#     stage="@MLOPS_DEMO_DB.PIPELINES.ML_CODE_STAGE",
#     packages=["torch", "xgboost", "snowflake-ml-python"]
# )
# def train_model():
#     # Full training pipeline runs on the compute pool
#     import xgboost as xgb
#     from snowflake.snowpark.context import get_active_session
#     
#     session = get_active_session()
#     data = session.table("MLOPS_DEMO_DB.RAW.CUSTOMERS").to_pandas()
#     
#     model = xgb.XGBClassifier(n_estimators=200)
#     model.fit(data[feature_cols], data["EXITED"])
#     
#     return model
#
# # Submit the job
# job = train_model()
# print(f"Job submitted: {job.id}")
# print(f"Status: {job.status()}")
#
# # Wait for completion
# result = job.result()

print("ML Jobs enable remote execution on dedicated compute pools.")
print("CPU pools: ML_CPU_POOL (CPU_X64_S)")
print("GPU pools: ML_GPU_POOL (GPU_NV_S)")
print("\nJobs integrate with Snowflake Tasks for scheduled execution.")

In [ ]:
# Check available compute pools
session.sql("SHOW COMPUTE POOLS").show()

## Summary: When to Use Each Approach

| Approach | Best For | Compute | Scale |
|----------|----------|---------|-------|
| **Snowflake ML Estimators** | Classical ML (XGBoost, LightGBM, sklearn) | Warehouse | Distributed across warehouse nodes |
| **GridSearchCV** | Hyperparameter tuning | Warehouse | Parallel CV folds across nodes |
| **PyTorchDistributor** | Deep learning (large models, large data) | GPU Compute Pool | Multi-node, multi-GPU DDP |
| **ML Jobs** | Any long-running training | CPU/GPU Compute Pool | Dedicated compute, async execution |
| **Container Runtime** | Interactive development | Notebook compute | Single node (CPU or GPU) |

## What to Show in SnowSight

Navigate to **Admin > Compute Pools** to see:
- Available compute pools (CPU and GPU)
- Pool status and utilization
- Instance families and sizes

Navigate to **Monitoring > Query History** to see:
- ML Job execution history
- Training job durations and resource usage

**Key message**: Snowflake supports the full spectrum of training workloads -- from classical ML on warehouses to distributed deep learning on GPU clusters -- all managed within the platform.